In [7]:
# Cell 1: Imports and Setup
from ipywidgets import FloatSlider
from IPython.display import display
import torch
import torch.nn as nn
import numpy as np
from deep_sdf import data, utils, mesh
import deep_sdf.workspace as ws
import json 
import os
import meshplot as mp
import trimesh
import sys
import networks.sdf_vae as vae
from networks.approximator_networks import LatentToHLLEICAApproximator, HLLEICAToLatentInverse
from finetune_encoder_decoder_with_approximator import SDFVAEWithApproximator, load_frozen_approximator_pipeline


In [2]:
# Cell 2: Load Fine-tuned Model
experiment_directory = "examples/torus_subgroup/only_pointnet_deep_sdf_no_KL"

print("🔄 Loading fine-tuned model with approximator pipeline...")

# Load specifications
specs_filename = os.path.join(experiment_directory, "specs.json")
if not os.path.isfile(specs_filename):
    raise Exception('The experiment directory does not include specifications file "specs.json"')

specs = json.load(open(specs_filename))

# Model parameters
latent_size = specs["CodeLength"]
num_samp_per_scene = specs["SamplesPerScene"]
decoder_specs = specs["NetworkSpecs"]
kl_div_loss = False

# Load the fine-tuned model
log_dir = os.path.join(experiment_directory, "finetune_with_approximator")
model_path = os.path.join(log_dir, "final_finetuned_model.pth")

if not os.path.exists(model_path):
    raise Exception(f"Fine-tuned model not found at {model_path}")

checkpoint = torch.load(model_path)

# Recreate the model architecture
original_sdfvae = vae.SDFVAE(latent_size, num_samp_per_scene, decoder_specs, kl_div_loss).cuda()
approximator, inverse_net, _ = load_frozen_approximator_pipeline(experiment_directory)

combined_model = SDFVAEWithApproximator(
    original_sdfvae, approximator, inverse_net, freeze_approximator=True
).cuda()

combined_model.load_state_dict(checkpoint['model_state_dict'])
combined_model.eval()

print("✅ Fine-tuned model loaded successfully!")


🔄 Loading fine-tuned model with approximator pipeline...
✅ Fine-tuned model loaded successfully!


In [3]:
# Cell 3: Helper Classes
class HiddenPrints:
    """Context manager to suppress print statements"""
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

class HLLEICAMeshGenerator:
    """Wrapper to generate meshes from HLLE+ICA space"""
    def __init__(self, combined_model, kl_div_loss):
        self.combined_model = combined_model
        self.kl_div_loss = kl_div_loss
        self.approximator = combined_model.approximator
        self.inverse_net = combined_model.inverse_net
        self.decoder = combined_model.decoder
    
    def generate_mesh_from_hlle_ica(self, hlle_ica_vector, N=64, max_batch=int(2**18)):
        """Generate mesh directly from HLLE+ICA space"""
        hlle_ica_vector = hlle_ica_vector.cuda()
        
        # Convert HLLE+ICA to latent space using frozen inverse network
        with torch.no_grad():
            reconstructed_latent = self.inverse_net(hlle_ica_vector.unsqueeze(0))
        
        # Use the reconstructed latent to generate mesh
        with torch.no_grad():
            # Create a bounding box for the mesh generation
            def implicit_func(points):
                points = points.cuda()
                batch_size = points.shape[0]
                
                # Expand latent for all query points
                latent_expanded = reconstructed_latent.expand(batch_size, -1)
                
                # Concatenate latent and points
                decoder_input = torch.cat([latent_expanded, points], dim=1)
                
                # Get SDF values
                sdf_values = self.decoder(decoder_input)
                return sdf_values.squeeze()
            
            # Use marching cubes to extract mesh
            generated_mesh = mesh.create_mesh_from_implicit(
                implicit_func,
                N=N,
                max_batch=max_batch,
                bbox_min=-1.5,
                bbox_max=1.5
            )
            
            return generated_mesh
    
    def __call__(self, hlle_ica_vector, N=64, max_batch=int(2**18)):
        return self.generate_mesh_from_hlle_ica(hlle_ica_vector, N, max_batch)

# Create mesh generator
mesh_generator = HLLEICAMeshGenerator(combined_model, kl_div_loss)


In [4]:
# Cell 4: Determine HLLE+ICA Value Ranges from Fine-tuned Approximator - FIXED
print("🔄 Computing HLLE+ICA value ranges from fine-tuned approximator model...")

# Load training data to get latents and generate HLLE+ICA embeddings
data_source = specs["DataSource"]
data_source_mesh = specs["DataSourceMesh"]
train_split_file = specs["TrainSplit"]

with open(train_split_file, "r") as f:
    import json
    train_split = json.load(f)

# Load some training samples to compute HLLE+ICA ranges from fine-tuned model
print("Loading training samples...")
num_samples_for_range = min(50, len(train_split))  # Use subset for efficiency
sample_filenames = train_split[:num_samples_for_range]

hlle_ica_embeddings_list = []

for i, filename in enumerate(sample_filenames):
    try:
        print(f"Processing {i+1}/{num_samples_for_range}: {filename}")
        
        # FIXED: Handle filename properly to avoid double .obj extension
        base_name = os.path.basename(filename)
        
        # Remove .npz extension if present, but keep .obj if already there
        if base_name.endswith(".npz"):
            mesh_name = base_name[:-4] + ".obj"
        elif base_name.endswith(".obj"):
            mesh_name = base_name  # Already has .obj extension
        else:
            mesh_name = base_name + ".obj"
        
        mesh_path = os.path.join(data_source_mesh, mesh_name)
        
        print(f"   Looking for mesh at: {mesh_path}")
        
        if os.path.exists(mesh_path):
            # Load surface points
            surface_points = data.get_surface_points(mesh_path)
            surface_points = torch.from_numpy(surface_points).unsqueeze(0).cuda()
            
            with torch.no_grad():
                # Encode surface points to get original latent
                if kl_div_loss:
                    mu, logvar = combined_model.encoder(surface_points)
                    z_original = mu  # Use mean for consistency
                else:
                    z_original = combined_model.encoder(surface_points)
                
                # Pass through fine-tuned approximator to get HLLE+ICA embedding
                hlle_ica_embedding = combined_model.approximator(z_original)
                hlle_ica_embeddings_list.append(hlle_ica_embedding.cpu().numpy())
                
            print(f"   ✅ Successfully processed {mesh_name}")
                
        else:
            print(f"   ❌ Mesh file not found: {mesh_path}")
            
            # Debug: List what files are actually in the mesh directory
            if i < 5:  # Only for first few files to avoid spam
                mesh_dir = os.path.dirname(mesh_path)
                if os.path.exists(mesh_dir):
                    available_files = [f for f in os.listdir(mesh_dir) if f.startswith(base_name.split('.')[0])]
                    if available_files:
                        print(f"   📁 Available similar files: {available_files[:3]}")
            
    except Exception as e:
        print(f"   ❌ Error processing {filename}: {str(e)}")
        continue

print(f"\n📊 Processing Summary:")
print(f"   Total attempted: {num_samples_for_range}")
print(f"   Successfully processed: {len(hlle_ica_embeddings_list)}")

if len(hlle_ica_embeddings_list) > 0:
    # Convert to numpy array
    hlle_ica_np = np.vstack(hlle_ica_embeddings_list).squeeze()
    print(f"✅ Successfully processed {len(hlle_ica_embeddings_list)} samples")
    print(f"HLLE+ICA embeddings shape: {hlle_ica_np.shape}")
    
    # Compute ranges for each HLLE+ICA component from fine-tuned approximator
    hlle_ica_ranges = {}
    hlle_ica_means = {}
    
    for i in range(6):  # 6 HLLE+ICA components
        min_val = np.min(hlle_ica_np[:, i])
        max_val = np.max(hlle_ica_np[:, i])
        mean_val = np.mean(hlle_ica_np[:, i])
        std_val = np.std(hlle_ica_np[:, i])
        
        # Add padding based on standard deviation for better exploration
        padding = max(std_val * 2, (max_val - min_val) * 0.2)
        hlle_ica_ranges[i] = (min_val - padding, max_val + padding)
        hlle_ica_means[i] = mean_val
        
        print(f"\nHLLE+ICA Component {i}:")
        print(f"  Range: [{min_val:.3f}, {max_val:.3f}]")
        print(f"  Mean: {mean_val:.3f}, Std: {std_val:.3f}")
        print(f"  Slider range: [{hlle_ica_ranges[i][0]:.3f}, {hlle_ica_ranges[i][1]:.3f}]")
    
    # Save the computed HLLE+ICA embeddings for reference
    hlle_ica_save_path = os.path.join(experiment_directory, "finetune_with_approximator", "hlle_ica_embeddings.npy")
    np.save(hlle_ica_save_path, hlle_ica_np)
    print(f"\n💾 Saved HLLE+ICA embeddings to: {hlle_ica_save_path}")
    
elif len(hlle_ica_embeddings_list) == 0:
    print("⚠️ No samples processed successfully!")
    print("   Checking if mesh directory exists...")
    
    # Debug the data source path
    print(f"   DataSourceMesh from specs: {data_source_mesh}")
    if os.path.exists(data_source_mesh):
        print(f"   ✅ Mesh directory exists")
        # Show some sample files
        mesh_files = [f for f in os.listdir(data_source_mesh)[:10] if f.endswith('.obj')]
        print(f"   📁 Sample mesh files: {mesh_files}")
    else:
        print(f"   ❌ Mesh directory does not exist!")
        # Check if there's a different mesh directory
        experiment_dir_files = os.listdir(experiment_directory) if os.path.exists(experiment_directory) else []
        print(f"   Available directories in experiment: {experiment_dir_files}")
    
    # Use default ranges
    print("   Using default ranges for HLLE+ICA components")
    hlle_ica_ranges = {i: (-2.0, 2.0) for i in range(6)}
    hlle_ica_means = {i: 0.0 for i in range(6)}
    
else:
    print("⚠️ Some samples processed successfully, but very few")
    print("   Using default ranges")
    hlle_ica_ranges = {i: (-2.0, 2.0) for i in range(6)}
    hlle_ica_means = {i: 0.0 for i in range(6)}

🔄 Computing HLLE+ICA value ranges from fine-tuned approximator model...
Loading training samples...
Processing 1/50: torus_bump_0505.obj
   Looking for mesh at: /home/jakaria/torus_bump_5000_two_scale_binary_bump_variable_noise_fixed_angle_two_subgroup_bump/scaled_obj_files/torus_bump_0505.obj
   ✅ Successfully processed torus_bump_0505.obj
Processing 2/50: torus_bump_3796.obj
   Looking for mesh at: /home/jakaria/torus_bump_5000_two_scale_binary_bump_variable_noise_fixed_angle_two_subgroup_bump/scaled_obj_files/torus_bump_3796.obj
   ✅ Successfully processed torus_bump_3796.obj
Processing 3/50: torus_bump_4084.obj
   Looking for mesh at: /home/jakaria/torus_bump_5000_two_scale_binary_bump_variable_noise_fixed_angle_two_subgroup_bump/scaled_obj_files/torus_bump_4084.obj
   ✅ Successfully processed torus_bump_4084.obj
Processing 4/50: torus_bump_1369.obj
   Looking for mesh at: /home/jakaria/torus_bump_5000_two_scale_binary_bump_variable_noise_fixed_angle_two_subgroup_bump/scaled_obj_fi

In [5]:
# Cell 5: Interactive HLLE+ICA Visualization with Fine-tuned Model
print("🎨 Creating HLLE+ICA interpolation interface using fine-tuned approximator...")

# Component names based on your HLLE+ICA analysis
component_names = [
    "Bump Presence",     # Component 0
    "Scale",             # Component 1  
    "Bump Size",         # Component 2
    "Subgroup",          # Component 3
    "Shape Detail 1",    # Component 4
    "Shape Detail 2"     # Component 5
]

# Initial HLLE+ICA vector (start from means)
hlle_ica_vector = torch.tensor([hlle_ica_means[i] for i in range(6)])
plot = None

@mp.interact(**{f'hlle_ica[{i}]': FloatSlider(
    min=hlle_ica_ranges[i][0], 
    max=hlle_ica_ranges[i][1], 
    step=(hlle_ica_ranges[i][1] - hlle_ica_ranges[i][0])/50, 
    value=hlle_ica_means[i],
    description=component_names[i][:12]  # Truncate for display
) for i in range(6)})
def show_hlle_ica_finetuned(**kwargs):
    global plot
    global hlle_ica_vector
    
    # Update HLLE+ICA vector
    hlle_ica_vector = torch.tensor([kwargs[f'hlle_ica[{i}]'] for i in range(6)])
    
    # Show current values
    print("Current HLLE+ICA values (from fine-tuned approximator):")
    for i in range(6):
        val = kwargs[f'hlle_ica[{i}]']
        deviation = val - hlle_ica_means[i]
        print(f"  {component_names[i]}: {val:.3f} (deviation: {deviation:+.3f})")
    
    try:
        # Generate mesh from HLLE+ICA space using fine-tuned inverse network
        with torch.no_grad():
            # Convert HLLE+ICA to latent space using fine-tuned inverse network
            reconstructed_latent = combined_model.inverse_net(hlle_ica_vector.unsqueeze(0).cuda())
            
            # Create mesh using custom mesh generation to avoid compatibility issues
            generated_mesh = create_mesh_from_latent_custom(
                reconstructed_latent.squeeze(), 
                combined_model.decoder, 
                N=96, 
                max_batch=int(2**18)
            )
        
        if generated_mesh is not None:
            # Extract vertices and faces
            verts = np.array(generated_mesh.vertices)
            faces = np.array(generated_mesh.faces)
            
            # Color based on the dominant component value
            dominant_component = np.argmax(np.abs([kwargs[f'hlle_ica[{i}]'] - hlle_ica_means[i] for i in range(6)]))
            
            # Create color based on dominant component
            colors = [
                [1.0, 0.7, 0.7],  # Light red for bump presence
                [0.7, 1.0, 0.7],  # Light green for scale
                [0.7, 0.7, 1.0],  # Light blue for bump size
                [1.0, 1.0, 0.7],  # Light yellow for subgroup
                [1.0, 0.7, 1.0],  # Light magenta for detail 1
                [0.7, 1.0, 1.0],  # Light cyan for detail 2
            ]
            
            mesh_color = np.array(colors[dominant_component])
            
            if plot is None:
                plot = mp.plot(verts, faces, c=mesh_color, return_plot=True)
            else:
                with HiddenPrints():
                    plot.update_object(vertices=verts, faces=faces)
                display(plot._renderer)
                
            print(f"✅ Mesh generated! Dominant component: {component_names[dominant_component]}")
            print(f"   Vertices: {len(verts)}, Faces: {len(faces)}")
            
            # Show reconstruction quality
            print(f"   Latent reconstruction shape: {reconstructed_latent.shape}")
            
        else:
            print("❌ Failed to generate mesh")
            
    except Exception as e:
        print(f"❌ Error generating mesh: {str(e)}")
        import traceback
        traceback.print_exc()

def create_mesh_from_latent_custom(latent_vector, decoder, N=64, max_batch=int(2**18)):
    """
    Custom mesh generation function for HLLE+ICA interpolation
    Bypasses compatibility issues with the existing mesh generation
    """
    # Set up voxel grid
    voxel_origin = [-1.2, -1.2, -1.2]  # Slightly larger for torus
    voxel_size = 2.4 / (N - 1)
    
    overall_index = torch.arange(0, N ** 3, 1, out=torch.LongTensor())
    samples = torch.zeros(N ** 3, 4)
    
    # Create coordinate grid
    samples[:, 2] = overall_index % N
    samples[:, 1] = (overall_index.long() / N) % N
    samples[:, 0] = ((overall_index.long() / N) / N) % N
    
    # Transform to world coordinates
    samples[:, 0] = (samples[:, 0] * voxel_size) + voxel_origin[2]
    samples[:, 1] = (samples[:, 1] * voxel_size) + voxel_origin[1]
    samples[:, 2] = (samples[:, 2] * voxel_size) + voxel_origin[0]
    
    num_samples = N ** 3
    samples.requires_grad = False
    head = 0
    
    # Evaluate SDF at all sample points
    while head < num_samples:
        sample_subset = samples[head : min(head + max_batch, num_samples), 0:3].cuda()
        
        # Create decoder input
        batch_size = sample_subset.shape[0]
        latent_expanded = latent_vector.expand(batch_size, -1)
        decoder_input = torch.cat([latent_expanded, sample_subset], dim=1)
        
        # Get SDF values
        sdf = decoder(decoder_input)
        
        samples[head : min(head + max_batch, num_samples), 3] = (
            sdf.squeeze().detach().cpu()
        )
        head += max_batch
    
    sdf_values = samples[:, 3].reshape(N, N, N)
    
    # Use marching cubes to generate the mesh
    try:
        import skimage.measure
        verts, faces, normals, values = skimage.measure.marching_cubes(
            sdf_values.numpy(), level=0.0, spacing=[voxel_size] * 3, method="lewiner"
        )
        
        # Transform verts to correct coordinate system
        mesh_points = np.zeros_like(verts)
        mesh_points[:, 0] = voxel_origin[0] + verts[:, 0]
        mesh_points[:, 1] = voxel_origin[1] + verts[:, 1]
        mesh_points[:, 2] = voxel_origin[2] + verts[:, 2]
        
        # Create trimesh object
        mesh = trimesh.Trimesh(vertices=mesh_points, faces=faces, process=False)
        return mesh
        
    except Exception as e:
        print(f"Marching cubes error: {e}")
        return None


🎨 Creating HLLE+ICA interpolation interface using fine-tuned approximator...


interactive(children=(FloatSlider(value=-0.010418075136840343, description='Bump Presenc', max=0.0432697990909…

In [8]:
# Cell 6: Frozen Model Visualization (Original Encoder/Decoder + Frozen Approximator)
print("🔄 Loading FROZEN model visualization (Original Encoder/Decoder + Frozen Approximator)...")

class SDFVAEWithFrozenApproximator(nn.Module):
    """
    Original SDFVAE with frozen approximator pipeline (NO fine-tuning):
    [Frozen Original Encoder] → [Frozen Approximator] → [Frozen Inverse] → [Frozen Original Decoder]
    """
    def __init__(self, original_sdfvae, approximator, inverse_net):
        super().__init__()
        
        # Use original components (all frozen)
        self.encoder = original_sdfvae.encoder
        self.decoder = original_sdfvae.decoder
        self.num_samp_per_scene = original_sdfvae.num_samp_per_scene
        self.latent_size = original_sdfvae.latent_size
        self.kl_div_loss = original_sdfvae.kl_div_loss
        
        # Add frozen approximator pipeline
        self.approximator = approximator
        self.inverse_net = inverse_net
        
        # Freeze ALL components
        for param in self.encoder.parameters():
            param.requires_grad = False
        for param in self.decoder.parameters():
            param.requires_grad = False
        for param in self.approximator.parameters():
            param.requires_grad = False
        for param in self.inverse_net.parameters():
            param.requires_grad = False
            
        # Set all to eval mode
        self.encoder.eval()
        self.decoder.eval()
        self.approximator.eval()
        self.inverse_net.eval()
    
    def forward(self, points, queries, train=False):
        """Forward pass through: Original Encoder → Approximator → Inverse → Original Decoder"""
        if points is not None:
            # Step 1: Encode with original frozen encoder
            if self.kl_div_loss:
                mu, logvar = self.encoder(points)
                z_original = mu  # Use mean for consistency
            else:
                z_original = self.encoder(points)
            
            # Step 2: Pass through frozen approximator (Latent → HLLE+ICA)
            hlle_ica_embedding = self.approximator(z_original)
            
            # Step 3: Pass through frozen inverse (HLLE+ICA → Latent)
            z_reconstructed = self.inverse_net(hlle_ica_embedding)
            
            # Step 4: Prepare for decoder
            batch_size = z_reconstructed.shape[0]
            num_queries = queries.shape[0]
            queries_per_batch = num_queries // batch_size
            z_expanded = z_reconstructed.unsqueeze(1).repeat(1, queries_per_batch, 1).view(-1, self.latent_size)
            
            # Step 5: Decode with original frozen decoder
            queries = queries.cuda()
            decoder_input = torch.cat([z_expanded, queries], dim=1)
            sdf = self.decoder(decoder_input)
            
            if self.kl_div_loss:
                return sdf, mu, logvar, z_original, z_reconstructed, hlle_ica_embedding
            else:
                return sdf, z_original, z_reconstructed, hlle_ica_embedding
        else:
            # Direct decoding without encoder
            sdf = self.decoder(queries)
            return sdf, None, None, None

# Create frozen model with original weights
print("🔄 Loading original model weights...")
original_model_path = os.path.join(experiment_directory, ws.model_params_subdir, "latest.pth")

if os.path.exists(original_model_path):
    # Load original SDFVAE with original weights
    original_sdfvae_frozen = vae.SDFVAE(latent_size, num_samp_per_scene, decoder_specs, kl_div_loss).cuda()
    original_checkpoint = torch.load(original_model_path)
    original_sdfvae_frozen.load_state_dict(original_checkpoint["model_state_dict"])
    
    # Create frozen model
    frozen_model = SDFVAEWithFrozenApproximator(
        original_sdfvae_frozen, 
        combined_model.approximator,  # Use same trained approximator
        combined_model.inverse_net    # Use same trained inverse
    ).cuda()
    frozen_model.eval()
    
    print("✅ Frozen model created successfully!")
    
    # Use same HLLE+ICA ranges from fine-tuned analysis for fair comparison
    print("🎨 Creating FROZEN HLLE+ICA interpolation interface...")
    
    # Initial HLLE+ICA vector (start from means)
    hlle_ica_vector_frozen = torch.tensor([hlle_ica_means[i] for i in range(6)])
    plot_frozen = None
    
    @mp.interact(**{f'frozen_hlle_ica[{i}]': FloatSlider(
        min=hlle_ica_ranges[i][0], 
        max=hlle_ica_ranges[i][1], 
        step=(hlle_ica_ranges[i][1] - hlle_ica_ranges[i][0])/50, 
        value=hlle_ica_means[i],
        description=f"FROZEN {component_names[i][:8]}"  # Truncate for display
    ) for i in range(6)})
    def show_frozen_hlle_ica(**kwargs):
        global plot_frozen
        global hlle_ica_vector_frozen
        
        # Update HLLE+ICA vector
        hlle_ica_vector_frozen = torch.tensor([kwargs[f'frozen_hlle_ica[{i}]'] for i in range(6)])
        
        # Show current values
        print("Current HLLE+ICA values (FROZEN MODEL - Original Encoder/Decoder):")
        for i in range(6):
            val = kwargs[f'frozen_hlle_ica[{i}]']
            deviation = val - hlle_ica_means[i]
            print(f"  {component_names[i]}: {val:.3f} (deviation: {deviation:+.3f})")
        
        try:
            # Generate mesh from HLLE+ICA space using frozen model
            with torch.no_grad():
                # Convert HLLE+ICA to latent space using frozen inverse network
                reconstructed_latent = frozen_model.inverse_net(hlle_ica_vector_frozen.unsqueeze(0).cuda())
                
                # Create mesh using custom mesh generation
                generated_mesh = create_mesh_from_latent_custom(
                    reconstructed_latent.squeeze(), 
                    frozen_model.decoder.module if hasattr(frozen_model.decoder, 'module') else frozen_model.decoder, 
                    N=96, 
                    max_batch=int(2**18)
                )
            
            if generated_mesh is not None:
                # Extract vertices and faces
                verts = np.array(generated_mesh.vertices)
                faces = np.array(generated_mesh.faces)
                
                # Color based on the dominant component value (different colors for frozen)
                dominant_component = np.argmax(np.abs([kwargs[f'frozen_hlle_ica[{i}]'] - hlle_ica_means[i] for i in range(6)]))
                
                # Different color scheme for frozen model (more muted colors)
                frozen_colors = [
                    [0.8, 0.5, 0.5],  # Muted red for bump presence
                    [0.5, 0.8, 0.5],  # Muted green for scale
                    [0.5, 0.5, 0.8],  # Muted blue for bump size
                    [0.8, 0.8, 0.5],  # Muted yellow for subgroup
                    [0.8, 0.5, 0.8],  # Muted magenta for detail 1
                    [0.5, 0.8, 0.8],  # Muted cyan for detail 2
                ]
                
                mesh_color = np.array(frozen_colors[dominant_component])
                
                if plot_frozen is None:
                    plot_frozen = mp.plot(verts, faces, c=mesh_color, return_plot=True)
                else:
                    with HiddenPrints():
                        plot_frozen.update_object(vertices=verts, faces=faces)
                    display(plot_frozen._renderer)
                    
                print(f"✅ FROZEN Mesh generated! Dominant component: {component_names[dominant_component]}")
                print(f"   Vertices: {len(verts)}, Faces: {len(faces)}")
                print(f"   Latent reconstruction shape: {reconstructed_latent.shape}")
                print(f"   🔒 Using ORIGINAL encoder/decoder (NOT fine-tuned)")
                
            else:
                print("❌ Failed to generate frozen mesh")
                
        except Exception as e:
            print(f"❌ Error generating frozen mesh: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print("\n" + "="*60)
    print("🔒 FROZEN MODEL COMPARISON")
    print("="*60)
    print("This uses:")
    print("  • Original encoder (NOT fine-tuned)")
    print("  • Original decoder (NOT fine-tuned)")  
    print("  • Frozen approximator pipeline")
    print("  • Same HLLE+ICA ranges for fair comparison")
    print("\nCompare with Cell 5 (fine-tuned) to see the difference!")
    print("="*60)
    
else:
    print(f"❌ Original model not found at: {original_model_path}")
    print("Available model files:")
    model_dir = os.path.join(experiment_directory, ws.model_params_subdir)
    if os.path.exists(model_dir):
        available_files = [f for f in os.listdir(model_dir) if f.endswith('.pth')]
        print(f"  {available_files}")
    else:
        print(f"  Model directory does not exist: {model_dir}")

🔄 Loading FROZEN model visualization (Original Encoder/Decoder + Frozen Approximator)...
🔄 Loading original model weights...
✅ Frozen model created successfully!
🎨 Creating FROZEN HLLE+ICA interpolation interface...


interactive(children=(FloatSlider(value=-0.010418075136840343, description='FROZEN Bump Pre', max=0.0432697990…


🔒 FROZEN MODEL COMPARISON
This uses:
  • Original encoder (NOT fine-tuned)
  • Original decoder (NOT fine-tuned)
  • Frozen approximator pipeline
  • Same HLLE+ICA ranges for fair comparison

Compare with Cell 5 (fine-tuned) to see the difference!


In [9]:
# Cell 8: HLLE+ICA Component Correlation Analysis with Ground Truth Labels
print("🔬 HLLE+ICA Component Correlation Analysis with Ground Truth Labels")
print("=" * 80)

if len(hlle_ica_embeddings_list) > 0:
    # Load ground truth labels from training data
    print("Loading ground truth labels for correlation analysis...")
    
    # Load labels from the centralized labels.pt file
    labels_path = os.path.join(data_source, "labels.pt")
    
    if os.path.exists(labels_path):
        print(f"Loading labels from {labels_path}")
        all_labels = torch.load(labels_path)
        
        # Extract parameters for each processed filename
        ground_truth_factors = []
        valid_indices = []
        
        for idx, filename in enumerate(sample_filenames[:len(hlle_ica_embeddings_list)]):
            # Extract base name without extension to match with label dict key
            base_name = os.path.splitext(os.path.basename(filename))[0]
            
            if base_name in all_labels:
                label = all_labels[base_name]
                # Extract all 4 factors: has_bump, scale, bump_size, subgroup
                factors = [
                    int(label[0].item()),      # has_bump (binary: 0 or 1)
                    float(label[2].item()),    # scale (continuous)
                    float(label[3].item()),    # bump_size (continuous)
                    float(label[4].item())     # subgroup (discrete: 1 or 2)
                ]
                ground_truth_factors.append(factors)
                valid_indices.append(idx)
        
        if len(ground_truth_factors) > 0:
            # Convert to numpy arrays
            ground_truth_factors = np.array(ground_truth_factors)
            valid_hlle_ica_embeddings = hlle_ica_np[valid_indices]
            
            print(f"✅ Loaded {len(ground_truth_factors)} samples with both HLLE+ICA and ground truth labels")
            print(f"Ground truth factors shape: {ground_truth_factors.shape}")
            print(f"HLLE+ICA embeddings shape: {valid_hlle_ica_embeddings.shape}")
            
            # Factor names
            factor_names = ['has_bump', 'scale', 'bump_size', 'subgroup']
            
            # Calculate comprehensive correlation matrix: 4 factors × 6 HLLE+ICA components
            print(f"\n📊 COMPREHENSIVE CORRELATION MATRIX")
            print("=" * 80)
            print("Each cell shows correlation between ground truth factor (row) and HLLE+ICA component (column)")
            print()
            
            # Create correlation matrix
            correlation_matrix = np.zeros((len(factor_names), 6))  # 4 factors x 6 HLLE+ICA components
            
            # Print header
            header = f"{'Factor':<12}"
            for comp_idx in range(6):
                header += f"{'Comp' + str(comp_idx+1):<10}"
            header += f"{'Best Component':<15} {'Max |Corr|':<12}"
            print(header)
            print("-" * len(header))
            
            # Calculate and display correlations
            factor_best_components = {}
            
            for f_idx, factor_name in enumerate(factor_names):
                factor_data = ground_truth_factors[:, f_idx]
                
                # Calculate correlations with all 6 HLLE+ICA components
                correlations = []
                row_str = f"{factor_name:<12}"
                
                for comp_idx in range(6):
                    comp_data = valid_hlle_ica_embeddings[:, comp_idx]
                    corr = np.corrcoef(comp_data, factor_data)[0, 1]
                    correlation_matrix[f_idx, comp_idx] = corr
                    correlations.append(corr)
                    
                    # Color coding for display
                    if abs(corr) >= 0.7:
                        color_symbol = "🟢"  # Strong correlation
                    elif abs(corr) >= 0.4:
                        color_symbol = "🟡"  # Moderate correlation
                    elif abs(corr) >= 0.2:
                        color_symbol = "🔶"  # Weak correlation
                    else:
                        color_symbol = "⚪"  # Very weak correlation
                    
                    row_str += f"{corr:+.3f} {color_symbol}  "
                
                # Find best component for this factor
                best_comp_idx = np.argmax(np.abs(correlations))
                best_corr = correlations[best_comp_idx]
                factor_best_components[factor_name] = {
                    'component_idx': best_comp_idx,
                    'component_name': component_names[best_comp_idx],
                    'correlation': best_corr
                }
                
                row_str += f"{'Comp' + str(best_comp_idx+1):<15} {abs(best_corr):.3f}"
                print(row_str)
            
            print()
            print("Legend: 🟢 Strong (≥0.7)  🟡 Moderate (≥0.4)  🔶 Weak (≥0.2)  ⚪ Very weak (<0.2)")
            
            # Summary of best mappings
            print(f"\n🏆 BEST HLLE+ICA COMPONENT FOR EACH FACTOR")
            print("=" * 60)
            for factor_name, best_info in factor_best_components.items():
                comp_idx = best_info['component_idx']
                comp_name = best_info['component_name']
                corr = best_info['correlation']
                
                # Interpretation
                if abs(corr) >= 0.7:
                    quality = "EXCELLENT"
                elif abs(corr) >= 0.5:
                    quality = "GOOD"
                elif abs(corr) >= 0.3:
                    quality = "MODERATE"
                else:
                    quality = "POOR"
                
                print(f"{factor_name:<12} → Component {comp_idx+1} ({comp_name})")
                print(f"             Correlation: {corr:+.4f} ({quality})")
                print()
            
            # Component utilization analysis
            print(f"🔍 COMPONENT UTILIZATION ANALYSIS")
            print("=" * 40)
            component_assignments = {}
            for comp_idx in range(6):
                assigned_factors = []
                for factor_name, best_info in factor_best_components.items():
                    if best_info['component_idx'] == comp_idx:
                        assigned_factors.append(factor_name)
                component_assignments[comp_idx] = assigned_factors
            
            for comp_idx in range(6):
                comp_name = component_names[comp_idx]
                assigned = component_assignments[comp_idx]
                if assigned:
                    print(f"Component {comp_idx+1} ({comp_name}): Controls {', '.join(assigned)}")
                else:
                    print(f"Component {comp_idx+1} ({comp_name}): Not strongly correlated with any factor")
            
            # Detailed correlation analysis for each component
            print(f"\n📈 DETAILED ANALYSIS FOR EACH HLLE+ICA COMPONENT")
            print("=" * 60)
            
            for comp_idx in range(6):
                comp_name = component_names[comp_idx]
                print(f"\nComponent {comp_idx+1} ({comp_name}):")
                
                # Get correlations for this component with all factors
                comp_correlations = correlation_matrix[:, comp_idx]
                
                # Sort factors by correlation strength
                factor_corr_pairs = [(factor_names[i], comp_correlations[i]) for i in range(len(factor_names))]
                factor_corr_pairs.sort(key=lambda x: abs(x[1]), reverse=True)
                
                for factor_name, corr in factor_corr_pairs:
                    strength = "Strong" if abs(corr) >= 0.5 else "Moderate" if abs(corr) >= 0.3 else "Weak"
                    print(f"  {factor_name:<12}: {corr:+.4f} ({strength})")
            
            # Quality metrics
            print(f"\n🎯 OVERALL QUALITY METRICS")
            print("=" * 30)
            
            # Calculate overall quality
            all_best_correlations = [abs(info['correlation']) for info in factor_best_components.values()]
            mean_correlation = np.mean(all_best_correlations)
            
            # Count strong correlations
            strong_correlations = sum(1 for corr in all_best_correlations if corr >= 0.5)
            moderate_correlations = sum(1 for corr in all_best_correlations if 0.3 <= corr < 0.5)
            
            print(f"Mean best correlation: {mean_correlation:.4f}")
            print(f"Strong correlations (≥0.5): {strong_correlations}/4")
            print(f"Moderate correlations (0.3-0.5): {moderate_correlations}/4")
            print(f"Overall interpretability: {mean_correlation:.1%}")
            
            # Component independence check
            print(f"\n🔄 COMPONENT INDEPENDENCE CHECK")
            print("=" * 35)
            hlle_ica_corr_matrix = np.corrcoef(valid_hlle_ica_embeddings.T)
            high_intercorr_pairs = []
            
            for i in range(6):
                for j in range(i+1, 6):
                    intercorr = abs(hlle_ica_corr_matrix[i, j])
                    if intercorr > 0.5:  # Components are too correlated
                        high_intercorr_pairs.append((i, j, intercorr))
            
            if high_intercorr_pairs:
                print("⚠️ Warning: High inter-component correlations detected:")
                for i, j, corr in high_intercorr_pairs:
                    print(f"  Component {i+1} ↔ Component {j+1}: {corr:.3f}")
            else:
                print("✅ Good component independence (all inter-correlations < 0.5)")
            
            # Save correlation results
            correlation_results = {
                'correlation_matrix': correlation_matrix.tolist(),
                'factor_names': factor_names,
                'component_names': component_names,
                'best_components_per_factor': factor_best_components,
                'mean_correlation': float(mean_correlation),
                'strong_correlations': int(strong_correlations),
                'moderate_correlations': int(moderate_correlations),
                'component_assignments': {f"component_{i+1}": component_assignments[i] for i in range(6)}
            }
            
            correlation_save_path = os.path.join(experiment_directory, "finetune_with_approximator", "hlle_ica_correlation_analysis.json")
            with open(correlation_save_path, 'w') as f:
                json.dump(correlation_results, f, indent=2)
            
            print(f"\n💾 Saved correlation analysis to: {correlation_save_path}")
            
        else:
            print("❌ No valid ground truth factors found for correlation analysis")
    else:
        print(f"❌ Labels file not found: {labels_path}")
        print("Available files in data source:")
        if os.path.exists(data_source):
            files = [f for f in os.listdir(data_source) if f.endswith('.pt')][:5]
            print(f"  {files}")
        
else:
    print("⚠️ No HLLE+ICA embeddings available for correlation analysis")

print("\n" + "=" * 80)

🔬 HLLE+ICA Component Correlation Analysis with Ground Truth Labels
Loading ground truth labels for correlation analysis...
Loading labels from /home/jakaria/torus_bump_5000_two_scale_binary_bump_variable_noise_fixed_angle_two_subgroup_bump/sdf_data/SdfSamples/scaled_obj_files/labels.pt
✅ Loaded 50 samples with both HLLE+ICA and ground truth labels
Ground truth factors shape: (50, 4)
HLLE+ICA embeddings shape: (50, 6)

📊 COMPREHENSIVE CORRELATION MATRIX
Each cell shows correlation between ground truth factor (row) and HLLE+ICA component (column)

Factor      Comp1     Comp2     Comp3     Comp4     Comp5     Comp6     Best Component  Max |Corr|  
----------------------------------------------------------------------------------------------------
has_bump    +0.166 ⚪  +0.224 🔶  -0.601 🟡  -0.300 🔶  +0.803 🟢  +0.517 🟡  Comp5           0.803
scale       +0.008 ⚪  -0.051 ⚪  -0.437 🟡  +0.799 🟢  -0.010 ⚪  +0.251 🔶  Comp4           0.799
bump_size   -0.094 ⚪  -0.043 ⚪  +0.023 ⚪  -0.041 ⚪  +0.036

TypeError: Object of type int64 is not JSON serializable

In [13]:
# Cell 8: HLLE+ICA Component Correlation Analysis - USING SAVED LATENTS (FIXED)
print("🔬 HLLE+ICA Component Correlation Analysis with Ground Truth Labels")
print("=" * 80)

# Load pre-computed latent codes instead of running networks
print("Loading pre-computed latent codes...")

# Option 1: Load from your experiment's saved latents
latents_path = os.path.join(experiment_directory, "metrics", "all_latents.npy")
if os.path.exists(latents_path):
    print(f"✅ Loading latents from: {latents_path}")
    all_latents_array = np.load(latents_path)
    if all_latents_array.ndim == 3:  # Shape like (400, 1, 16)
        all_latents_array = all_latents_array.squeeze(1)  # Convert to (400, 16)
    print(f"Loaded latents shape: {all_latents_array.shape}")
else:
    # Option 2: Load from workspace latent codes directory
    latest_latents_path = os.path.join(experiment_directory, ws.latent_codes_subdir, "latest.pth")
    if os.path.exists(latest_latents_path):
        print(f"✅ Loading latents from: {latest_latents_path}")
        latent_data = torch.load(latest_latents_path)
        if isinstance(latent_data["latent_codes"], dict):
            # Extract embedding weights
            all_latents_array = latent_data["latent_codes"]["weight"].cpu().numpy()
        else:
            all_latents_array = latent_data["latent_codes"].cpu().numpy()
        print(f"Loaded latents shape: {all_latents_array.shape}")
    else:
        raise Exception(f"No saved latents found at {latents_path} or {latest_latents_path}")

# Load ground truth labels
labels_path = os.path.join(data_source, "labels.pt")
if not os.path.exists(labels_path):
    raise Exception(f"Labels file not found: {labels_path}")

print(f"Loading labels from {labels_path}")
all_labels = torch.load(labels_path)

# Load training split to match indices
with open(train_split_file, "r") as f:
    train_split = json.load(f)

# Extract ground truth factors for each sample
ground_truth_factors = []
valid_latents = []
valid_filenames = []

for idx, filename in enumerate(train_split):
    # Extract base name without extension to match with label dict key
    base_name = os.path.splitext(os.path.basename(filename))[0]
    
    if base_name in all_labels and idx < len(all_latents_array):
        label = all_labels[base_name]
        # Extract all 4 factors: has_bump, scale, bump_size, subgroup
        factors = [
            int(label[0].item()),      # has_bump (binary: 0 or 1)
            float(label[2].item()),    # scale (continuous)
            float(label[3].item()),    # bump_size (continuous)
            float(label[4].item())     # subgroup (discrete: 1 or 2)
        ]
        ground_truth_factors.append(factors)
        valid_latents.append(all_latents_array[idx])
        valid_filenames.append(filename)

# Convert to numpy arrays
ground_truth_factors = np.array(ground_truth_factors)
valid_latents = np.array(valid_latents)

print(f"✅ Matched {len(ground_truth_factors)} samples with both latents and ground truth labels")
print(f"Latent codes shape: {valid_latents.shape}")
print(f"Ground truth factors shape: {ground_truth_factors.shape}")

# Compute REAL HLLE+ICA directly from saved latent codes
print("\n🔄 Computing REAL HLLE+ICA directly from saved latent codes...")

try:
    from sklearn.manifold import LocallyLinearEmbedding
    from sklearn.decomposition import FastICA
    from sklearn.preprocessing import StandardScaler
    
    # Standardize the latent codes
    scaler = StandardScaler()
    latent_codes_scaled = scaler.fit_transform(valid_latents)
    
    # Step 1: Apply HLLE (Hessian LLE) to latent codes
    print("   Applying HLLE (Hessian Locally Linear Embedding)...")
    
    # FIXED: Calculate proper n_neighbors for Hessian LLE
    n_components = 6
    min_neighbors_required = int(n_components * (n_components + 3) / 2) + 1  # +1 for safety
    n_neighbors = min(50, len(valid_latents) - 1)  # Use 50 neighbors (safe for 400 samples)
    
    if n_neighbors < min_neighbors_required:
        print(f"   ⚠️ Warning: Need at least {min_neighbors_required} neighbors for 6 components")
        print(f"   Available samples: {len(valid_latents)}, using {n_neighbors} neighbors")
        # Reduce components if not enough neighbors
        n_components = min(6, int(np.sqrt(2 * (n_neighbors - 1))))
        print(f"   Reducing to {n_components} components")
    
    print(f"   Using {n_neighbors} neighbors for {n_components} HLLE components")
    
    hlle = LocallyLinearEmbedding(
        n_neighbors=n_neighbors,
        n_components=n_components,
        eigen_solver='dense',
        method='hessian',
        random_state=42
    )
    
    hlle_embedding = hlle.fit_transform(latent_codes_scaled)
    print(f"   ✅ HLLE embedding shape: {hlle_embedding.shape}")
    
    # Step 2: Apply ICA to HLLE embedding
    print("   Applying ICA (Independent Component Analysis)...")
    ica = FastICA(
        n_components=n_components,  # Use same number as HLLE
        whiten='arbitrary-variance',
        random_state=42,
        max_iter=2000,
        tol=1e-6
    )
    
    real_hlle_ica = ica.fit_transform(hlle_embedding)
    print(f"   ✅ Real HLLE+ICA shape: {real_hlle_ica.shape}")
    
    # Factor names
    factor_names = ['has_bump', 'scale', 'bump_size', 'subgroup']
    
    # Calculate comprehensive correlation matrix: 4 factors × n_components HLLE+ICA components
    print(f"\n📊 COMPREHENSIVE CORRELATION MATRIX (Real HLLE+ICA from Saved Latents)")
    print("=" * 80)
    print("Each cell shows correlation between ground truth factor (row) and REAL HLLE+ICA component (column)")
    print()
    
    # Create correlation matrix
    correlation_matrix = np.zeros((len(factor_names), n_components))  # 4 factors x n_components HLLE+ICA components
    
    # Print header
    header = f"{'Factor':<12}"
    for comp_idx in range(n_components):
        header += f"{'Comp' + str(comp_idx+1):<10}"
    header += f"{'Best Component':<15} {'Max |Corr|':<12}"
    print(header)
    print("-" * len(header))
    
    # Calculate and display correlations
    factor_best_components = {}
    
    for f_idx, factor_name in enumerate(factor_names):
        factor_data = ground_truth_factors[:, f_idx]
        
        # Calculate correlations with all HLLE+ICA components
        correlations = []
        row_str = f"{factor_name:<12}"
        
        for comp_idx in range(n_components):
            comp_data = real_hlle_ica[:, comp_idx]
            corr = np.corrcoef(comp_data, factor_data)[0, 1]
            correlation_matrix[f_idx, comp_idx] = corr
            correlations.append(corr)
            
            # Color coding for display
            if abs(corr) >= 0.7:
                color_symbol = "🟢"  # Strong correlation
            elif abs(corr) >= 0.4:
                color_symbol = "🟡"  # Moderate correlation
            elif abs(corr) >= 0.2:
                color_symbol = "🔶"  # Weak correlation
            else:
                color_symbol = "⚪"  # Very weak correlation
            
            row_str += f"{corr:+.3f} {color_symbol}  "
        
        # Find best component for this factor
        best_comp_idx = np.argmax(np.abs(correlations))
        best_corr = correlations[best_comp_idx]
        factor_best_components[factor_name] = {
            'component_idx': best_comp_idx,
            'component_name': f"Real Component {best_comp_idx+1}",
            'correlation': best_corr
        }
        
        row_str += f"{'Comp' + str(best_comp_idx+1):<15} {abs(best_corr):.3f}"
        print(row_str)
    
    print()
    print("Legend: 🟢 Strong (≥0.7)  🟡 Moderate (≥0.4)  🔶 Weak (≥0.2)  ⚪ Very weak (<0.2)")
    
    # Summary of best mappings
    print(f"\n🏆 BEST REAL HLLE+ICA COMPONENT FOR EACH FACTOR")
    print("=" * 60)
    for factor_name, best_info in factor_best_components.items():
        comp_idx = best_info['component_idx']
        comp_name = best_info['component_name']
        corr = best_info['correlation']
        
        # Interpretation
        if abs(corr) >= 0.7:
            quality = "EXCELLENT"
        elif abs(corr) >= 0.5:
            quality = "GOOD"
        elif abs(corr) >= 0.3:
            quality = "MODERATE"
        else:
            quality = "POOR"
        
        print(f"{factor_name:<12} → Component {comp_idx+1}")
        print(f"             Correlation: {corr:+.4f} ({quality})")
        print()
    
    # Quality metrics
    print(f"\n🎯 OVERALL REAL HLLE+ICA QUALITY METRICS")
    print("=" * 40)
    
    # Calculate overall quality
    all_best_correlations = [abs(info['correlation']) for info in factor_best_components.values()]
    mean_correlation = np.mean(all_best_correlations)
    
    # Count strong correlations
    strong_correlations = sum(1 for corr in all_best_correlations if corr >= 0.5)
    moderate_correlations = sum(1 for corr in all_best_correlations if 0.3 <= corr < 0.5)
    
    print(f"Mean best correlation: {mean_correlation:.4f}")
    print(f"Strong correlations (≥0.5): {strong_correlations}/4")
    print(f"Moderate correlations (0.3-0.5): {moderate_correlations}/4")
    print(f"Overall interpretability: {mean_correlation:.1%}")
    
    # Save correlation results
    correlation_results = {
        'real_hlle_ica_correlation_matrix': correlation_matrix.tolist(),
        'factor_names': factor_names,
        'best_components_per_factor': factor_best_components,
        'mean_correlation': float(mean_correlation),
        'strong_correlations': int(strong_correlations),
        'moderate_correlations': int(moderate_correlations),
        'real_hlle_ica_embeddings': real_hlle_ica.tolist(),
        'latent_codes': valid_latents.tolist(),
        'sample_filenames': valid_filenames,
        'n_components': n_components,
        'n_neighbors': n_neighbors
    }
    
    correlation_save_path = os.path.join(experiment_directory, "finetune_with_approximator", "real_hlle_ica_correlation_analysis_from_saved.json")
    with open(correlation_save_path, 'w') as f:
        json.dump(correlation_results, f, indent=2)
    
    print(f"\n💾 Saved real HLLE+ICA correlation analysis to: {correlation_save_path}")
    
except ImportError:
    print("❌ sklearn not available - cannot compute real HLLE+ICA")
    print("   Install with: pip install scikit-learn")
except Exception as e:
    print(f"❌ Error computing real HLLE+ICA: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)

🔬 HLLE+ICA Component Correlation Analysis with Ground Truth Labels
Loading pre-computed latent codes...
✅ Loading latents from: examples/torus_subgroup/only_pointnet_deep_sdf_no_KL/metrics/all_latents.npy
Loaded latents shape: (400, 16)
Loading labels from /home/jakaria/torus_bump_5000_two_scale_binary_bump_variable_noise_fixed_angle_two_subgroup_bump/sdf_data/SdfSamples/scaled_obj_files/labels.pt
✅ Matched 400 samples with both latents and ground truth labels
Latent codes shape: (400, 16)
Ground truth factors shape: (400, 4)

🔄 Computing REAL HLLE+ICA directly from saved latent codes...
   Applying HLLE (Hessian Locally Linear Embedding)...
   Using 50 neighbors for 6 HLLE components
   ✅ HLLE embedding shape: (400, 6)
   Applying ICA (Independent Component Analysis)...
   ✅ Real HLLE+ICA shape: (400, 6)

📊 COMPREHENSIVE CORRELATION MATRIX (Real HLLE+ICA from Saved Latents)
Each cell shows correlation between ground truth factor (row) and REAL HLLE+ICA component (column)

Factor      

/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/site-packages/sklearn/decomposition/_fastica.py:128: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
Traceback (most recent call last):
  File "/tmp/ipykernel_192501/2365401714.py", line 245, in <module>
    json.dump(correlation_results, f, indent=2)
  File "/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/json/__init__.py", line 179, in dump
    for chunk in iterable:
  File "/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/json/encoder.py", line 431, in _iterencode
    yield from _iterencode_dict(o, _current_indent_level)
  File "/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/json/encoder.py", line 405, in _iterencode_dict
    yield from chunks
  File "/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/json/encoder.py", line 405, in _iterencode_dict
    yield from chunks
  File "/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/json/enco